# Month-end finance analysis
Revenue, DSO trend, collections vs billing, and overdue accounts. Run from this directory. Amounts assume a single reporting currency; timestamps use the warehouse date. Rolling windows include the current partial month.

In [ ]:
from finance_nb.warehouse import connect, run_query

config = {
    "host": "tdprod-fin.otterworks.internal",
    "user": "nb_finance_analysis",
    "password": "not-a-real-password",
    "database": "FIN_DW",
}
connection = connect(config)

## Revenue by month

In [ ]:
revenue_sql = """SEL (INVOICE_DT (FORMAT 'YYYY-MM') (CHAR(7))) AS MONTH_LABEL,
  SUM(AMOUNT_DUE - ZEROIFNULL(TAX_AMT)) AS REVENUE
FROM FIN_DW.INVOICE
WHERE INVOICE_DT >= ADD_MONTHS(CURRENT_DATE, -12) AND INVOICE_DT >= 1200101
GROUP BY 1 ORDER BY 1"""
revenue = run_query(connection, revenue_sql)
revenue

## DSO trend

In [ ]:
dso_sql = """WITH billed AS (
  SEL (INVOICE_DT (FORMAT 'YYYY-MM') (CHAR(7))) AS MONTH_LABEL,
    SUM(AMOUNT_DUE) AS BILLED
  FROM FIN_DW.INVOICE WHERE INVOICE_DT >= ADD_MONTHS(CURRENT_DATE, -12)
    AND INVOICE_DT >= 1200101 GROUP BY 1
), open_ar AS (
  SEL b.MONTH_LABEL, SUM(i.AMOUNT_DUE - ZEROIFNULL(p.PAID)) AS OPEN_AR
  FROM billed b JOIN FIN_DW.INVOICE i
    ON i.INVOICE_DT <= LAST_DAY(CAST(b.MONTH_LABEL || '-01' AS DATE FORMAT 'YYYY-MM-DD'))
  LEFT JOIN (
    SEL b.MONTH_LABEL, p.INVOICE_ID, SUM(p.AMOUNT) AS PAID
    FROM billed b JOIN FIN_DW.PAYMENT p
      ON p.PAYMENT_DT <= LAST_DAY(CAST(b.MONTH_LABEL || '-01' AS DATE FORMAT 'YYYY-MM-DD'))
    GROUP BY 1, 2
  ) p ON p.INVOICE_ID = i.INVOICE_ID AND p.MONTH_LABEL = b.MONTH_LABEL
  WHERE i.INVOICE_DT >= 1200101 GROUP BY 1
)
SEL b.MONTH_LABEL, ZEROIFNULL(a.OPEN_AR / NULLIFZERO(b.BILLED) * 30) AS DSO
FROM billed b JOIN open_ar a ON a.MONTH_LABEL = b.MONTH_LABEL ORDER BY 1"""
dso = run_query(connection, dso_sql)
dso

## Collections vs billing

In [ ]:
collections_sql = """WITH billed AS (
  SEL (INVOICE_DT (FORMAT 'YYYY-MM') (CHAR(7))) AS MONTH_LABEL,
    SUM(AMOUNT_DUE) AS BILLED
  FROM FIN_DW.INVOICE WHERE INVOICE_DT >= CURRENT_DATE - 90
    AND INVOICE_DT >= 1200101 GROUP BY 1
), cash AS (
  SEL (PAYMENT_DT (FORMAT 'YYYY-MM') (CHAR(7))) AS MONTH_LABEL,
    SUM(AMOUNT) AS COLLECTED
  FROM FIN_DW.PAYMENT WHERE PAYMENT_DT >= CURRENT_DATE - 90 GROUP BY 1
)
SEL COALESCE(b.MONTH_LABEL, c.MONTH_LABEL) AS MONTH_LABEL,
  ZEROIFNULL(b.BILLED) AS BILLED, ZEROIFNULL(c.COLLECTED) AS COLLECTED
FROM billed b FULL OUTER JOIN cash c ON c.MONTH_LABEL = b.MONTH_LABEL ORDER BY 1"""
collections = run_query(connection, collections_sql)
collections

## Top overdue accounts

In [ ]:
overdue_sql = """WITH balances AS (
  SEL i.ACCOUNT_ID, i.DUE_DT, i.AMOUNT_DUE - ZEROIFNULL(p.PAID) AS OPEN_BALANCE
  FROM FIN_DW.INVOICE i
  LEFT JOIN (SEL INVOICE_ID, SUM(AMOUNT) AS PAID FROM FIN_DW.PAYMENT GROUP BY INVOICE_ID) p
    ON p.INVOICE_ID = i.INVOICE_ID
  WHERE i.INVOICE_DT >= 1200101 AND i.DUE_DT < CURRENT_DATE - 30
)
SEL TOP 10 a.ACCOUNT_NO, c.CUSTOMER_NAME,
  CAST(MIN(b.DUE_DT) AS DATE FORMAT 'YYYY-MM-DD') AS OLDEST_DUE_DATE,
  SUM(b.OPEN_BALANCE) AS OVERDUE_BALANCE
FROM balances b JOIN FIN_DW.ACCOUNT a ON a.ACCOUNT_ID = b.ACCOUNT_ID
JOIN FIN_DW.CUSTOMER c ON c.CUSTOMER_ID = a.CUSTOMER_ID
WHERE b.OPEN_BALANCE > 0 GROUP BY a.ACCOUNT_NO, c.CUSTOMER_NAME
ORDER BY OVERDUE_BALANCE DESC, a.ACCOUNT_NO"""
overdue = run_query(connection, overdue_sql)
overdue

In [ ]:
connection.close()